In [ ]:
import os
import numpy as np
import torch
import cv2
import rawpy

from debayer import Debayer5x5, Layout 

def postprocess(rgb):
    rgb = rgb.cpu().detach().numpy().astype(np.float32)
    rgb = np.transpose(rgb, (0,2,3,1))
    rgb = np.clip(rgb, 0, 1)
    rgb = np.uint8(rgb*255)
    return rgb

debayer = Debayer5x5(layout=Layout.GBRG).cuda(device=1)

def raw_to_png(raw_file_path, save_folder):
    savename = os.path.basename(raw_file_path).replace('.raw', '.png')
    
    with open(raw_file_path, 'rb') as f:
        raw = np.fromfile(f, dtype=np.uint16)

    
    #raw = raw.reshape(512, 512)  
    #raw = raw.reshape(2160, 3840)
    raw = raw.reshape(3000, 4000)
    #raw = raw.reshape(896, 448)
    #raw = rawpy.imread(raw_file_path).raw_image_visible
    #raw = raw.reshape(2848, 4256)
    
    raw = raw.astype(np.float32)
    raw = (raw - 256) / ((2**12 - 1) - 256)
    #raw = (raw-512) / ((2**16 - 1)-512)
    #raw = raw / 65535.0
    raw = raw * 20.0 #gain
    raw = torch.from_numpy(raw).unsqueeze(0).unsqueeze(0).cuda(device=1)
    #raw = SpaceToDepth_fact2(raw)

    with torch.no_grad():
        aa = torch.clamp(raw, min=0.0, max=1.0)
        aa = torch.clamp(aa, min=1e-8) ** (1.0 / 2.2)

        aa = postprocess(debayer(aa.clone().detach()))[0]
        
        cv2.imwrite(os.path.join(save_folder, savename), cv2.cvtColor(aa, cv2.COLOR_BGR2RGB))

    print(f'Saved PNG and RAW files for {savename}')

raw_file_path ="/database2/iyj0121/samsung/SAMSUNG_TESTSET/s4/SN_20250903_231954_02_4000x3000.raw"
save_folder = "/database2/iyj0121/samsung/SAMSUNG_TESTSET/s4/"

if not os.path.exists(save_folder):
    os.makedirs(save_folder)

raw_to_png(raw_file_path, save_folder)


Saved PNG and RAW files for SN_20250903_231953_01_4000x3000.png


In [6]:
import numpy as np
import cv2
from PIL import Image
import glob
import os

HEIGHT = 3000
WIDTH = 4000
BLACK_LEVEL = 256
WHITE_LEVEL = 4095

def unpack_raw12_packed(path, height, width):
    data = np.fromfile(path, dtype=np.uint8)

    expected_bytes = height * width * 12 // 8
    data = data.reshape(-1, 3)

    # MIPI RAW12 packed
    p0 = (data[:, 0].astype(np.uint16) << 4) | (data[:, 2].astype(np.uint16) & 0x0F)
    p1 = (data[:, 1].astype(np.uint16) << 4) | (data[:, 2].astype(np.uint16) >> 4)

    raw = np.empty(p0.size + p1.size, dtype=np.uint16)
    raw[0::2] = p0
    raw[1::2] = p1

    return raw.reshape(height, width)

def load_dataset(folder):
    files = glob.glob(os.path.join(folder, "*.raw"))
    files = sorted(files, key=lambda x: os.path.basename(x))
    black = BLACK_LEVEL
    white = WHITE_LEVEL
    return files, black, white

def readRaw(path, black, white, height=HEIGHT, width=WIDTH):
    raw = unpack_raw12_packed(path, height, width).astype(np.float32)

    raw = np.maximum(raw - black, 0) / (white - black)

    if raw.ndim == 2:
        h, w = raw.shape
        raw = raw.reshape(h, w, 1)

    return raw

def gains():
    rgb_gain = 1.0
    red_gain = 2.002930
    blue_gain = 1.657227
    return rgb_gain, red_gain, blue_gain

def apply_wb_bayer(raw, rgb_gain, red_gain, blue_gain, pattern="GRBG"):
    raw = raw.copy()

    if pattern == "RGGB":
        raw[0::2, 0::2] *= red_gain * rgb_gain
        raw[0::2, 1::2] *= rgb_gain
        raw[1::2, 0::2] *= rgb_gain
        raw[1::2, 1::2] *= blue_gain * rgb_gain

    elif pattern == "GBRG":
        raw[0::2, 0::2] *= rgb_gain
        raw[0::2, 1::2] *= blue_gain * rgb_gain
        raw[1::2, 0::2] *= red_gain * rgb_gain
        raw[1::2, 1::2] *= rgb_gain

    elif pattern == "GRBG":
        raw[0::2, 0::2] *= rgb_gain
        raw[0::2, 1::2] *= red_gain * rgb_gain
        raw[1::2, 0::2] *= blue_gain * rgb_gain
        raw[1::2, 1::2] *= rgb_gain

    else:
        raise ValueError(f"Unsupported pattern: {pattern}")

    return np.clip(raw, 0.0, 1.0)

def raw_to_srgb(raw_bayer, gamma=2.2, apply_wb=True, exposure=1.0):
    # raw_bayer
    if raw_bayer.ndim == 3:
        raw = raw_bayer[..., 0]
    else:
        raw = raw_bayer

    raw = raw.astype(np.float32)
    raw = np.clip(raw, 0.0, 1.0)

    # WB in Bayer domain
    if apply_wb:
        rgb_gain, red_gain, blue_gain = gains()
        raw = apply_wb_bayer(raw, rgb_gain, red_gain, blue_gain)

    # exposure
    raw = np.clip(raw * exposure, 0.0, 1.0)

    # demosaic (GBRG)
    raw_16 = (raw * 65535.0).astype(np.uint16)
    rgb = cv2.cvtColor(raw_16, cv2.COLOR_BAYER_GR2RGB)

    # [0,1]
    rgb = rgb.astype(np.float32) / 65535.0

    # gamma
    rgb = np.clip(rgb, 0.0, 1.0)
    rgb = np.power(rgb, 1.0 / gamma)

    # uint8
    rgb_8 = (rgb * 255.0).clip(0, 255).astype(np.uint8)
    return rgb_8

input_path = "/database/iyj0121/dataset/4k_vnt/lowlight_orin/"
save_dir = "/database/iyj0121/dataset/4k_vnt/lowlight_orin/sRGB"
os.makedirs(save_dir, exist_ok=True)

raw_paths, black, white = load_dataset(input_path)

print(f"Raw file: {len(raw_paths)}")
print(f"black={black}, white={white}")

for raw_path in raw_paths:
    stem = os.path.splitext(os.path.basename(raw_path))[0]
    save_file = os.path.join(save_dir, f"{stem}.jpg")

    raw_bayer = readRaw(raw_path, black=black, white=white, height=HEIGHT, width=WIDTH)
    srgb = raw_to_srgb(
        raw_bayer,
        gamma=3.5,
        apply_wb=False,
        exposure=2.0,
    )

    srgb = np.rot90(srgb, k=3)

    Image.fromarray(srgb).save(save_file)
    print(f"Saved {save_file}")

Raw file: 11
black=256, white=4095
Saved /database/iyj0121/dataset/4k_vnt/lowlight_orin/sRGB/SN_20251205_140246_00_4000x3000.jpg
Saved /database/iyj0121/dataset/4k_vnt/lowlight_orin/sRGB/SN_20251205_140246_01_4000x3000.jpg
Saved /database/iyj0121/dataset/4k_vnt/lowlight_orin/sRGB/SN_20251205_140246_02_4000x3000.jpg
Saved /database/iyj0121/dataset/4k_vnt/lowlight_orin/sRGB/SN_20251205_140247_03_4000x3000.jpg
Saved /database/iyj0121/dataset/4k_vnt/lowlight_orin/sRGB/SN_20251205_140247_04_4000x3000.jpg
Saved /database/iyj0121/dataset/4k_vnt/lowlight_orin/sRGB/SN_20251205_140247_05_4000x3000.jpg
Saved /database/iyj0121/dataset/4k_vnt/lowlight_orin/sRGB/SN_20251205_140247_06_4000x3000.jpg
Saved /database/iyj0121/dataset/4k_vnt/lowlight_orin/sRGB/SN_20251205_140247_07_4000x3000.jpg
Saved /database/iyj0121/dataset/4k_vnt/lowlight_orin/sRGB/SN_20251205_140247_08_4000x3000.jpg
Saved /database/iyj0121/dataset/4k_vnt/lowlight_orin/sRGB/SN_20251205_140247_09_4000x3000.jpg
Saved /database/iyj0121/d

In [4]:
import os
import numpy as np
import torch
import cv2

from debayer import Debayer5x5, Layout


def postprocess(rgb):
    rgb = rgb.detach().cpu().numpy().astype(np.float32)
    rgb = np.transpose(rgb, (0, 2, 3, 1))
    rgb = np.clip(rgb, 0, 1)
    rgb = (rgb * 255).astype(np.uint8)
    return rgb


debayer = Debayer5x5(layout=Layout.GBRG).cuda(device=1)


def unpack_raw12_packed(raw_bytes, width, height):
    expected_size = width * height * 3 // 2
    if raw_bytes.size != expected_size:
        raise ValueError(
            f"RAW12 packed size mismatch: got {raw_bytes.size}, expected {expected_size}"
        )

    b = raw_bytes.reshape(-1, 3).astype(np.uint16)

    # 일반적인 MIPI RAW12 packed 해석
    p0 = (b[:, 0] << 4) | (b[:, 2] & 0x0F)
    p1 = (b[:, 1] << 4) | (b[:, 2] >> 4)

    raw = np.empty(p0.size + p1.size, dtype=np.uint16)
    raw[0::2] = p0
    raw[1::2] = p1

    raw = raw.reshape(height, width)
    return raw


def raw_to_png(raw_file_path, save_folder, width=4000, height=3000):
    savename = os.path.basename(raw_file_path).replace(".raw", ".png")

    raw_bytes = np.fromfile(raw_file_path, dtype=np.uint8)
    raw = unpack_raw12_packed(raw_bytes, width, height)

    raw = raw.astype(np.float32)

    black_level = 256
    white_level = 4095

    raw = np.clip(raw - black_level, 0, None) / (white_level - black_level)
    raw = raw * 2.0
    raw = np.clip(raw, 0.0, 1.0)

    raw = torch.from_numpy(raw).unsqueeze(0).unsqueeze(0).cuda(device=1)

    with torch.no_grad():
        aa = torch.clamp(raw, min=1e-8) ** (1.0 / 2.2)
        aa = postprocess(debayer(aa))[0]

        # aa는 RGB일 가능성이 높으므로 OpenCV 저장용으로 BGR 변환
        cv2.imwrite(os.path.join(save_folder, savename), cv2.cvtColor(aa, cv2.COLOR_RGB2BGR))

    print(f"Saved: {os.path.join(save_folder, savename)}")


raw_file_path = "/database/iyj0121/dataset/4k_vnt/lowlight_orin/SN_20251205_140246_00_4000x3000.raw"
save_folder = "/database/iyj0121/dataset/4k_vnt/lowlight_orin/"

os.makedirs(save_folder, exist_ok=True)
raw_to_png(raw_file_path, save_folder)

Saved: /database/iyj0121/dataset/4k_vnt/lowlight_orin/SN_20251205_140246_00_4000x3000.png


In [8]:
import glob

def all_raw_to_png(raw_folder_path, save_folder):
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)

    raw_files = sorted(glob.glob(os.path.join(raw_folder_path, '*.raw')))
    
    for raw_file_path in raw_files:
        raw_to_png(raw_file_path, save_folder)

raw_folder_path ="/database/iyj0121/dataset/4k_vnt/DPC_Dark_Moving_Chart_Dynamic_finetune_noise_estimation_crvd_method_neif_on_4k_calibration_gain_6_30_hcg_scene_from_1_to_6_video_model_size_up/"
save_folder ="/database/iyj0121/dataset/4k_vnt/DPC_Dark_Moving_Chart_Dynamic_finetune_noise_estimation_crvd_method_neif_on_4k_calibration_gain_6_30_hcg_scene_from_1_to_6_video_model_size_up_png_g100/"

if not os.path.exists(save_folder):
    os.makedirs(save_folder)

all_raw_to_png(raw_folder_path, save_folder)

Saved PNG and RAW files for denoised_raw_frame_000000.png
Saved PNG and RAW files for denoised_raw_frame_000001.png
Saved PNG and RAW files for denoised_raw_frame_000002.png
Saved PNG and RAW files for denoised_raw_frame_000003.png
Saved PNG and RAW files for denoised_raw_frame_000004.png
Saved PNG and RAW files for denoised_raw_frame_000005.png
Saved PNG and RAW files for denoised_raw_frame_000006.png
Saved PNG and RAW files for denoised_raw_frame_000007.png
Saved PNG and RAW files for denoised_raw_frame_000008.png
Saved PNG and RAW files for denoised_raw_frame_000009.png
Saved PNG and RAW files for denoised_raw_frame_000010.png
Saved PNG and RAW files for denoised_raw_frame_000011.png
Saved PNG and RAW files for denoised_raw_frame_000012.png
Saved PNG and RAW files for denoised_raw_frame_000013.png
Saved PNG and RAW files for denoised_raw_frame_000014.png
Saved PNG and RAW files for denoised_raw_frame_000015.png
Saved PNG and RAW files for denoised_raw_frame_000016.png
Saved PNG and 

In [18]:
import numpy as np
import os

for frame_ind in range(1, 7+1): 
    input_folder = f"/database/iyj0121/dataset/4k_vnt/VNT_241120_pair_video/scene5/scene5_frame{frame_ind}_6db/"
    output_folder = f"/database/iyj0121/dataset/4k_vnt/VNT_241120_pair_video/scene5/scene5_frame{frame_ind}_6db/"
    output_file = "cleaned_bayer_frame.raw"

    file_list = sorted([os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith('.raw')])
    print(len(file_list))

    if len(file_list) == 0:
        print(f"No RAW files found in {input_folder}. Skipping frame {frame_ind}.")
        continue

    sample_file = file_list[0]
    image_shape = (2160, 3840)

    sum_image = np.zeros(image_shape, dtype=np.float64)

    for file_path in file_list:
        image = np.fromfile(file_path, dtype=np.uint16).reshape(image_shape)
        sum_image += image

    mean_image = sum_image / len(file_list)

    mean_image = mean_image.astype(np.uint16)

    output_path = os.path.join(output_folder, output_file)
    mean_image.tofile(output_path)

    print(f"Frame {frame_ind}의 클린한 이미지가 {output_path}에 저장되었습니다.")


261
Frame 1의 클린한 이미지가 /database/iyj0121/dataset/4k_vnt/VNT_241120_pair_video/scene5/scene5_frame1_6db/cleaned_bayer_frame.raw에 저장되었습니다.
260
Frame 2의 클린한 이미지가 /database/iyj0121/dataset/4k_vnt/VNT_241120_pair_video/scene5/scene5_frame2_6db/cleaned_bayer_frame.raw에 저장되었습니다.
261
Frame 3의 클린한 이미지가 /database/iyj0121/dataset/4k_vnt/VNT_241120_pair_video/scene5/scene5_frame3_6db/cleaned_bayer_frame.raw에 저장되었습니다.
261
Frame 4의 클린한 이미지가 /database/iyj0121/dataset/4k_vnt/VNT_241120_pair_video/scene5/scene5_frame4_6db/cleaned_bayer_frame.raw에 저장되었습니다.
261
Frame 5의 클린한 이미지가 /database/iyj0121/dataset/4k_vnt/VNT_241120_pair_video/scene5/scene5_frame5_6db/cleaned_bayer_frame.raw에 저장되었습니다.
261
Frame 6의 클린한 이미지가 /database/iyj0121/dataset/4k_vnt/VNT_241120_pair_video/scene5/scene5_frame6_6db/cleaned_bayer_frame.raw에 저장되었습니다.
261
Frame 7의 클린한 이미지가 /database/iyj0121/dataset/4k_vnt/VNT_241120_pair_video/scene5/scene5_frame7_6db/cleaned_bayer_frame.raw에 저장되었습니다.


In [ ]:
exit()

: 

In [1]:
import numpy as np
import os

input_folder = f"/database/iyj0121/dataset/4k_vnt/VNT/"
output_folder = f"/database/iyj0121/dataset/4k_vnt/"
output_file = "VNT_cleaned_bayer_frame.raw"

file_list = sorted([os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith('.raw')])
print(len(file_list))

if len(file_list) == 0:
    print(f"No RAW files found in {input_folder}. Skipping frame {frame_ind}.")

sample_file = file_list[0]
image_shape = (2160, 3840)

sum_image = np.zeros(image_shape, dtype=np.float64)

for file_path in file_list:
    image = np.fromfile(file_path, dtype=np.uint16).reshape(image_shape)
    sum_image += image

mean_image = sum_image / len(file_list)

mean_image = mean_image.astype(np.uint16)

output_path = os.path.join(output_folder, output_file)
mean_image.tofile(output_path)

print(f"Frame의 클린한 이미지가 {output_path}에 저장되었습니다.")


270
Frame의 클린한 이미지가 /database/iyj0121/dataset/4k_vnt/VNT_cleaned_bayer_frame.raw에 저장되었습니다.


In [17]:
import os
import shutil

source_dir = "/database/iyj0121/dataset/4k_vnt/VNT_241120_pair_video/scene5/scene5_frame2_6db/"
target_dir = "/database/iyj0121/dataset/4k_vnt/VNT_241120_pair_video/scene5/scene5_frame2_6db_target/"

os.makedirs(target_dir, exist_ok=True)

files = [f for f in os.listdir(source_dir) if f.startswith("dump_bayer_frame_") and f.endswith(".raw")]
files.sort()

counter = 0
num_files = len(files)

while counter < 260:
    for file in files:
        if counter >= 260:
            break
        src_path = os.path.join(source_dir, file)
        new_filename = f"dump_bayer_frame_{counter:05d}.raw"
        dst_path = os.path.join(target_dir, new_filename)
        shutil.copy(src_path, dst_path)
        counter += 1

print(f"Successfully copied and renamed files to {target_dir}")


Successfully copied and renamed files to /database/iyj0121/dataset/4k_vnt/VNT_241120_pair_video/scene5/scene5_frame2_6db_target/


In [ ]:
exit()

: 